In [1]:
library(openxlsx)
library(stringr)
path <- "/scratch/user/uqqvo1/Medulloblastoma_analysis/scripts/"
work_dir <- dirname(path)
work_dir <- str_split(work_dir, 'scripts/')[[1]][1]
setwd(work_dir)


source('scripts/X2_DEAnalysis/clusterProfiler_helpers.R')

data_dir <- 'data/DE_out/Pseudo_Limma_human/'
out_dir <- 'data/DE_out/Pseudo_Limma_human/gsea_out/'
out_plots <- 'figure_components/DE_figures/'


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
  # Performing GSEA enrichment analysis #
################################################################################
# Using mSigDB as the database, see here:
#  http://yulab-smu.top/clusterProfiler-book/chapter3.html#msigdb-analysis

# 7.2.0 for msigdbr to work with vissE
library(stringr)
library(clusterProfiler)
library(enrichplot)
library(openxlsx)
library(ggplot2)
library(vissE)
library(GSEABase)
library(msigdb)

species <- "human"
suffix <- "human"

######### Loading the reference gene sets ###############
msigdbr_show_species()

db_t2g <- load_db(species)



Registered S3 methods overwritten by 'treeio':
  method              from    
  MRCA.phylo          tidytree
  MRCA.treedata       tidytree
  Nnode.treedata      tidytree
  Ntip.treedata       tidytree
  ancestor.phylo      tidytree
  ancestor.treedata   tidytree
  child.phylo         tidytree
  child.treedata      tidytree
  full_join.phylo     tidytree
  full_join.treedata  tidytree
  groupClade.phylo    tidytree
  groupClade.treedata tidytree
  groupOTU.phylo      tidytree
  groupOTU.treedata   tidytree
  is.rooted.treedata  tidytree
  nodeid.phylo        tidytree
  nodeid.treedata     tidytree
  nodelab.phylo       tidytree
  nodelab.treedata    tidytree
  offspring.phylo     tidytree
  offspring.treedata  tidytree
  parent.phylo        tidytree
  parent.treedata     tidytree
  root.treedata       tidytree
  rootnode.phylo      tidytree
  sibling.phylo       tidytree

clusterProfiler v4.6.2  For help: https://yulab-smu.top/biomedical-knowledge-mining-book/

If you use clusterProf

[1] "Anolis carolinensis"             "Bos taurus"                     
 [3] "Caenorhabditis elegans"          "Canis lupus familiaris"         
 [5] "Danio rerio"                     "Drosophila melanogaster"        
 [7] "Equus caballus"                  "Felis catus"                    
 [9] "Gallus gallus"                   "Homo sapiens"                   
[11] "Macaca mulatta"                  "Monodelphis domestica"          
[13] "Mus musculus"                    "Ornithorhynchus anatinus"       
[15] "Pan troglodytes"                 "Rattus norvegicus"              
[17] "Saccharomyces cerevisiae"        "Schizosaccharomyces pombe 972h-"
[19] "Sus scrofa"                      "Xenopus tropicalis"

Joining with `by = join_by(gs_cat, gs_subcat, gs_name, gene_symbol,
entrez_gene, ensembl_gene, human_gene_symbol, human_entrez_gene,
human_ensembl_gene, gs_id, gs_pmid, gs_geoid, gs_exact_source, gs_url,
gs_description)`


In [3]:
############ Loading the query gene sets ####################
file_name <- paste(data_dir, 'de_results_PseudoLimma_', suffix, '.xlsx', sep='')
geneList <- load_geneList(file_name)

In [4]:
############ Performing GSEA ####################
em <- GSEA(geneList, TERM2GENE = db_t2g, by='fgsea',
           minGSSize = 20, pvalueCutoff = .01, eps = 0)

preparing geneSet collections...

GSEA analysis...

leading edge analysis...

done...



In [5]:
# Adding in the percentage #
counts <- apply(em@result, 1, function(vals) {length(str_split(vals['core_enrichment'], '/')[[1]])})
perc <- round((counts / em@result[,'setSize'])*100, 1)
em@result['count'] <- counts
em@result['Percentage'] <- perc

result <- em@result
result <- result[order(result[,'NES'], decreasing = T),]
sets <- rownames(result)
print(sets[str_detect(sets, 'HALLMARK')])
print(length(sets))

##### Writing out the results #######
save_gseaResults(em, result, out_dir, suffix)

[1] "HALLMARK_MYOGENESIS"       "HALLMARK_MTORC1_SIGNALING"
[3] "HALLMARK_SPERMATOGENESIS"  "HALLMARK_DNA_REPAIR"      
[5] "HALLMARK_MYC_TARGETS_V1"   "HALLMARK_MITOTIC_SPINDLE" 
[7] "HALLMARK_G2M_CHECKPOINT"   "HALLMARK_E2F_TARGETS"     
[1] 249


Warning message:
“Please use 'rowNames' instead of 'row.names'”
Warning message:
“Please use 'colNames' instead of 'col.names'”


In [6]:
# Visualising the results #
################################################################################
file_name <- paste(out_dir, 'gsea_results_', suffix, '.rds', sep='')
em <- readRDS(file_name)
###### NOTE I flipped the NES score here so upregulation would be red to be consistent
###### in the figure, & changed the figure legend in the illustrator file to reflect this.
em@result[,'nes'] <- -em@result[,'NES']
result <- em@result
result <- result[order(result[,'NES'], decreasing = T),]

######### ClusterProfiler approach ##########
selected_terms <- c("GOBP_NEUROGENESIS", "GOBP_NEURON_DIFFERENTIATION", 
                    "GOBP_NEURON_DEVELOPMENT","DNA_REPAIR","GOBP_REGULATION_OF_SECRETION",
                    "HALLMARK_MYC_TARGETS_V1", "GOBP_CELL_DIVISION", "GOBP_DNA_REPLICATION",
                    "GOBP_CELL_CYCLE", "GOBP_HORMONE_TRANSPORT","REGULATION_OF_NEUROTRANSMITTER_LEVELS",
                    "HALLMARK_G2M_CHECKPOINT", "HALLMARK_E2F_TARGETS","NEUROTRANSMITTER_TRANSPORT",
                   "NEUROTRANSMITTER_SECRETION","ION_TRANSMEMBRANE_TRANSPORT")
selected_terms <- format_go_names(selected_terms)

em_formatted <- em
rownames(em_formatted@result) <- format_go_names(rownames(em_formatted@result))
em_formatted@result$ID <- rownames(em_formatted@result)
em_formatted@result$Description <- rownames(em_formatted@result)
names(em_formatted@geneSets) <- rownames(em_formatted@result)

In [7]:
# the dotplot of the GSEA results #
p <- dotplot(em_formatted, x="GeneRatio", showCategory=selected_terms, size='Percentage',
             font.size=22) + 
        theme(panel.grid.major = element_blank(), panel.grid.minor = element_blank(),
              text=element_text(face='bold', size=20), panel.border = element_blank(),
              axis.line = element_line())
plot_name <- paste(out_plots, species, suffix, 'GSEAdotplot.pdf', sep='_')
pdf(plot_name, width=8)
print(p)
dev.off()

png 
  2

In [8]:
# The emap plot #
edo <- pairwise_termsim(em_formatted)

In [9]:
p <- emapplot(edo, 
              showCategory = selected_terms, 
              pie_scale = 1, 
              repel = TRUE,
              cex.params = list(category_label = 1.2), 
              layout.params = list(layout = "nicely"), 
              color = 'nes') +
     theme(text = element_text(face = 'bold', size = 1))

# Setting the font face to bold for the third layer
p$layers[[3]]$aes_params[['fontface']] <- "bold"

# Define the plot name
plot_name <- paste(out_plots, species, suffix, 'GSEAemap.pdf', sep = '_')

# Save the plot as a PDF
pdf(plot_name)
print(p)
dev.off()


png 
  2